In [1]:
import pathlib

import pandas as pd
import numpy as np

import h5py

In [5]:
event_file = pathlib.Path(R"D:\datasets\own_data\mason_bees_1\raw\cam_0\event_file_uncorrected.hdf5")
with h5py.File(event_file, "r") as f_orig:
    dataset = f_orig["EXT_TRIGGER"]["events"]
    data = dataset[:]
    df = pd.DataFrame(data)
    orig_dtype_events = dataset.dtype

    orig_dtype_indexes = f_orig["EXT_TRIGGER"]["indexes"].dtype
    orig_chunks_indexes = f_orig["EXT_TRIGGER"]["indexes"].chunks
    orig_maxshape_indexes = f_orig["EXT_TRIGGER"]["indexes"].maxshape
    offset_indexes = int(f_orig["EXT_TRIGGER"]["indexes"].attrs.get("offset", 0))


In [6]:
df

,p,t,id
0,0,3068725,0
1,1,3074609,0
2,0,3074650,0
3,1,3080534,0
4,0,3080575,0
...,...,...,...
2078,0,11624377,0
2079,1,11630260,0
2080,0,11630302,0
2081,1,11636185,0


In [7]:
prev_time_low = 0
prev_time_high = 0
time_between_pol = df.iloc[2]["t"] - df.iloc[0]["t"]
margin_size = 10
total_missing_triggers = 0
print(f"Time between same polarity {time_between_pol}\n")

missing_triggers = []

for index, row in df.iterrows():
    if (index == 0 or index == 1) and row['p'] == 0:
        prev_time_low = row['t']
    if (index == 0 or index == 1) and row['p'] == 1:
        prev_time_high = row['t']

    if row['p'] == 0 and index > 1:
        time_diff = row['t'] - prev_time_low
        if time_diff >= (time_between_pol + margin_size):
            print(f"Incorrect timing {time_diff}, index {index}, polarity {row['p']}")
            missing_triggers_count = int(np.round(time_diff / time_between_pol)) - 1

            for i in range(missing_triggers_count):
                missing_triggers.append([0, prev_time_low + ((i+1) * time_between_pol), 0])

            total_missing_triggers += missing_triggers_count
            print(f"Estimated missing polarity 0 triggers {missing_triggers_count}\n")
        elif time_diff <= (time_between_pol - margin_size):
            print("Extra trigger detected, duplicates?")

        prev_time_low = row['t']

    if row['p'] == 1 and index > 1:
        time_diff = row['t'] - prev_time_high
        if time_diff >= (time_between_pol + margin_size):
            print(f"incorrect timing {time_diff}, index {index}, polarity {row['p']}")
            missing_triggers_count = int(np.round(time_diff / time_between_pol)) - 1

            for i in range(missing_triggers_count):
                missing_triggers.append([1, prev_time_high + ((i+1) * time_between_pol), 0])

            total_missing_triggers += missing_triggers_count
            print(f"Estimated missing polarity 1 triggers {missing_triggers_count}\n")
        elif time_diff <= (time_between_pol - margin_size):
            print("Extra trigger detected, duplicates?")

        prev_time_high = row['t']

print(f"Total missing triggers {total_missing_triggers}")


Time between same polarity 5925

incorrect timing 225149, index 193, polarity 1
Estimated missing polarity 1 triggers 37

Incorrect timing 225149, index 194, polarity 0
Estimated missing polarity 0 triggers 37

incorrect timing 118499, index 221, polarity 1
Estimated missing polarity 1 triggers 19

Incorrect timing 118499, index 222, polarity 0
Estimated missing polarity 0 triggers 19

incorrect timing 124424, index 249, polarity 1
Estimated missing polarity 1 triggers 20

Incorrect timing 124424, index 250, polarity 0
Estimated missing polarity 0 triggers 20

incorrect timing 124424, index 563, polarity 1
Estimated missing polarity 1 triggers 20

Incorrect timing 124424, index 564, polarity 0
Estimated missing polarity 0 triggers 20

incorrect timing 118500, index 591, polarity 1
Estimated missing polarity 1 triggers 19

Incorrect timing 118499, index 592, polarity 0
Estimated missing polarity 0 triggers 19

incorrect timing 681371, index 619, polarity 1
Estimated missing polarity 1 t

In [19]:
df_corr = pd.concat([df, pd.DataFrame(missing_triggers, columns=["p", "t", "id"])], axis=0).sort_values(by=["t"]).reset_index(drop=True)

Check if there are triggers that have the same polarity after each other

In [20]:
prev_time_pol = 0

for index, row in df_corr.iterrows():
    if (index == 0) and row['p'] == 0:
        prev_time_pol = row['p']
    if (index == 0) and row['p'] == 1:
        prev_time_pol = row['p']

        if index > 0 and prev_time_pol == row['p']:
            print("Double same polarity")
            prev_time_pol = row['p']

In [64]:
index_size = 2000

# Recreate index table
def build_indexes(events, offset, idx_dtype):
    shifted_t = events["t"].astype(np.int64) + offset

    last_bucket = np.ceil(shifted_t.to_numpy()[-1] / index_size)
    bucket_starts = np.arange(0, (last_bucket) * index_size, index_size, dtype=np.int64)

    out = np.empty(len(bucket_starts) + 1, dtype=idx_dtype)

    # First row
    out[0]["id"] = 0
    out[0]["ts"] = -1

    prev_id = 0
    prev_ts = shifted_t[0]

    for i, q in enumerate(bucket_starts, start=1):
        q_end = q + index_size

        # First event in [q, q_end)
        first = np.searchsorted(shifted_t, q, side="left")

        if first < len(shifted_t) and shifted_t[first] < q_end:
            prev_id = first
            prev_ts = shifted_t[first]

        out[i]["id"] = prev_id
        out[i]["ts"] = prev_ts

    return out

In [65]:
indexes = build_indexes(df_corr, offset_indexes, orig_dtype_indexes)

In [66]:
data_np = np.zeros(len(df_corr), dtype=orig_dtype_events)

for name in orig_dtype_events.names:
    data_np[name] = df_corr[name].values

with h5py.File(event_file.parent / "event_file.hdf5", "w") as f_corr, h5py.File(event_file) as f_orig:
    for key in f_orig.keys():
        f_orig.copy(key, f_corr)

    del f_corr["EXT_TRIGGER"]["events"]
    del f_corr["EXT_TRIGGER"]["indexes"]

    f_corr["EXT_TRIGGER"].create_dataset(
        "events",
        data=data_np,
        dtype=orig_dtype_events,
        shape=(len(data_np),),
        maxshape=(None,),
        chunks=(16384,),
    )

    f_corr["EXT_TRIGGER"].create_dataset(
        "indexes",
        data=indexes,
        dtype=orig_dtype_indexes,
        shape=(len(indexes),),
        maxshape=(None,),
        chunks=(16384,),
    )
    f_corr["EXT_TRIGGER"]["indexes"].attrs.create(
        "offset",
        np.bytes_(str(offset_indexes)),
        dtype=h5py.string_dtype(encoding="ascii")
    )

    # Copy root attrs
    for k, v in f_orig.attrs.items():
        f_corr.attrs[k] = v

    f_corr.close()